# Episode narration - Maya 2, voice from config.yaml

Pushed by `python -m abpipe.episode_tts push`. Narrates the episode's locked
Hinglish chapters and leaves per-chapter mp3s, measured timing maps and the
hash of every text it read in the output. Installs, then probes speed, then
narrates: a failure at either of the first two costs minutes, not a session.

In [ ]:
!pip -q install soundfile pyyaml
!apt-get -qq install -y ffmpeg > /dev/null
!nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv

In [ ]:
# maya3 pins torch 2.8.0 and transformers 5.6.2, so it gets its own venv. It
# lives in /tmp, not /kaggle/working: anything in working becomes kernel Output
# and is downloaded with it. The commit is pinned so a result stays attributable
# to the code that produced it.
import subprocess, sys, time

MAYA_SHA = "c6b376868e268551d065c4206193e96257bef4a1"
VENV = "/tmp/maya-venv"
REPO = "/tmp/maya3"

# Kaggle's image moved to Python 3.13; maya3 documents 3.10-3.12. uv fetches its
# own 3.12, so the venv does not depend on whatever the image ships next.
t0 = time.time()
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
subprocess.run([sys.executable, "-m", "uv", "venv", "-q", "--seed", "--python", "3.12", VENV], check=True)
subprocess.run(["git", "clone", "-q", "https://github.com/MayaResearch/maya2-native", REPO], check=True)
subprocess.run(["git", "-C", REPO, "checkout", "-q", MAYA_SHA], check=True)
subprocess.run([f"{VENV}/bin/pip", "install", "-q", "-e", REPO,
                "--extra-index-url", "https://download.pytorch.org/whl/cu128"], check=True)
print("venv built in %.1f min" % ((time.time() - t0) / 60))

# The T4 is sm_75. A torch without it fails here, in seconds, rather than after
# the 5 GB download.
CHECK = ("import torch, maya3; "
         "print('torch', torch.__version__, 'cuda', torch.version.cuda); "
         "print('arch', torch.cuda.get_arch_list()); "
         "print('gpus', torch.cuda.device_count(), torch.cuda.get_device_name(0)); "
         "print('bf16 native', torch.cuda.is_bf16_supported()); "
         "print('tensor', (torch.ones(4, device='cuda', dtype=torch.bfloat16) * 2).sum().item()); "
         "assert 'sm_75' in torch.cuda.get_arch_list()")
subprocess.run([f"{VENV}/bin/python", "-c", CHECK], check=True)

In [ ]:
import os, shutil, sys
from pathlib import Path

def _find_dataset(slug: str) -> Path:
    candidates = [Path('/kaggle/input') / slug,
                  *Path('/kaggle/input/datasets').glob(f'*/{slug}')]
    for c in candidates:
        if c.exists():
            return c
    raise FileNotFoundError(f"dataset '{slug}' not found under /kaggle/input")

os.environ['HF_TOKEN'] = (_find_dataset('hf-token') / 'HF_TOKEN').read_text().strip()

SRC  = _find_dataset('@DATASET@')
ROOT = Path('/kaggle/working/audiobook')
ROOT.mkdir(parents=True, exist_ok=True)
shutil.copytree(SRC / 'hindi', ROOT / 'hindi', dirs_exist_ok=True)
shutil.copy(SRC / 'manifest.json', ROOT / 'manifest.json')
shutil.copytree(SRC / 'abpipe', '/kaggle/working/abpipe', dirs_exist_ok=True)
sys.path.insert(0, '/kaggle/working')

import yaml
cfg = yaml.safe_load((SRC / 'config.yaml').read_text(encoding='utf-8'))
cfg['paths']['root'] = str(ROOT)
cfg['paths']['glossary'] = str(ROOT / 'glossary.json')
cfg['tts']['backend'] = 'maya2'
cfg['tts']['maya2']['venv_python'] = f'{VENV}/bin/python'
# The audiobook's 64k is fine for an m4b; this audio goes into a video.
cfg['output']['bitrate'] = '128k'
print(sorted(p.name for p in SRC.iterdir()))
cfg['tts']['maya2']

In [ ]:
import time
import soundfile as sf
from abpipe import tts as S
from abpipe.spoken_numbers import speakable

MAX_RTF = 10

t0 = time.time()
probe = S.make_backend(cfg['tts'])
print('probe load %.1f min' % ((time.time() - t0) / 60))
text = speakable('चैप्टर 1: फॉरेनर्स. लुमियान ने टाइम देखा, 7 बज चुके थे, और उसे अचानक एक आइडिया आया।')
wav = probe.say(text)   # first call: includes any kernel compilation
t0 = time.time()
wav = probe.say(text)
took = time.time() - t0
rtf = took / (len(wav) / probe.sample_rate)
print(f'probe {len(wav) / probe.sample_rate:.1f}s audio in {took:.1f}s  RTF {rtf:.2f}')
sf.write('/kaggle/working/probe.wav', wav, probe.sample_rate)
probe.close()
assert rtf <= MAX_RTF, f'RTF {rtf:.1f} is too slow to attempt the episode'

In [ ]:
import hashlib, json
from abpipe.manifest import Manifest
from abpipe.runner import run_batch

# Every staged chapter, read from the manifest - never a hard-coded count.
run_batch(cfg, len(Manifest.load(ROOT / 'manifest.json').chapters))

# No assert: a chapter that failed still leaves the kernel COMPLETE, so the
# ones that worked can be pulled. `episode_tts pull` reports the failures.
for ch in Manifest.load(ROOT / 'manifest.json').chapters:
    print(ch.idx, ch.status, ch.seconds, ch.rtf, (ch.error or '')[-300:])
hashes = {f.name: hashlib.sha256(f.read_bytes()).hexdigest()
          for f in sorted((ROOT / 'hindi').glob('*.txt'))}
(ROOT / 'hashes.json').write_text(json.dumps(hashes, indent=1))

In [ ]:
!rm -rf /kaggle/working/abpipe
!ls -la /kaggle/working/audiobook/audio /kaggle/working/audiobook/timing